In [ ]:
## TRANSCRIPTOMICS DATA ANALYSIS USING CODEX PYTHON PACAKAGES.

In [ ]:
# IMPORT PACKAGES

import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
# SET MATRIX PATH
## MATRIX LOOKS LIKE THIS :
### Rows = genes
### Columns = cells
### gene | cell_1 | cell_2 | cell_3 ...

matrix_path = r"data/processed/GSE150903_SCT_scaled_count_matrix.txt"

In [ ]:
# COUNT GENES AND CELLS

with open(matrix_path, "r") as f:
    header = f.readline().strip().split("\t")

n_cells = len(header) - 1
n_genes = sum(1 for _ in open(matrix_path, "r")) - 1

print("Number of cells:", n_cells)
print("Number of genes:", n_genes)

In [ ]:
# LOAD MATRIX AND CREATE ANNDATA
## SCANPY NEEDS CELLS IN ROWS AND GENES IN COLUMNS - THIS IS WHY WE TRANSPOSE!
counts = pd.read_csv(matrix_path, sep="\t", index_col=0)

print("Original shape:", counts.shape)  # genes x cells

counts_t = counts.T

print("Transposed shape:", counts_t.shape)  # cells x genes

adata = sc.AnnData(counts_t)

adata

In [ ]:
# ADD SAMPLE METADATA

adata.obs["prefix"] = adata.obs_names.str.split("_").str[0]

sample_map = {
    "T": "Telencephalon organoids",
    "1": "Choroid Plexus Org D27",
    "2": "Choroid Plexus Org D46",
    "3": "Choroid Plexus Org D53",
}

adata.obs["sample"] = adata.obs["prefix"].map(sample_map)

adata.obs.head()

In [ ]:
# SAVE RAW ANNDATA
## AT THIS POINT WE HAVE CONVERTED: GSE150903_SCT_scaled_count_matrix.txt -> gse150903_raw_counts.h5ad

raw_output_path = r"data/processed/gse150903_raw_counts.h5ad"

adata.write_h5ad(raw_output_path)

print("Saved:", raw_output_path)


In [ ]:
## END OF GENERATING ANNDATA
## NEXT STEP: QC 
### n_genes_by_counts, total_counts, pct_counts_mt

In [ ]:
# QC
## n_genes_by_counts = how many genes were detected in each cell
## total_counts      = total expression/counts in each cell
## pct_counts_mt     = percent mitochondrial gene expression

import scanpy as sc
import matplotlib.pyplot as plt

# Load the raw AnnData object
adata = sc.read_h5ad(
    r"data/processed/gse150903_raw_counts.h5ad"
)

# Mark mitochondrial genes.
# In human data, mitochondrial genes usually start with "MT-".
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Calculate quality-control metrics for each cell.
# This adds columns to adata.obs:
## total_counts, n_genes_by_counts, pct_counts_mt
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True
)

# Show summary statistics for the main QC metrics
display(
    adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()
)

# Show median QC values by sample
display(
    adata.obs
    .groupby("sample")[["n_genes_by_counts", "total_counts", "pct_counts_mt"]]
    .median()
)

In [ ]:
## QC PLOTS

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="sample",
    rotation=45,
    multi_panel=True
)

sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt"
)

In [ ]:
# QC DETECTS LOW QUALITY CELLS OR DATA POINTS THAT MAY BE EXCLUDED FROM FURTHER ANALYSIS. THIS IS DONE PRIOR TO NORMALISATION.

# Save counts before filtering
n_cells_before = adata.n_obs
n_genes_before = adata.n_vars

# Filter cells with too few detected genes.
# These may be empty droplets or poor-quality cells.
adata = adata[adata.obs["n_genes_by_counts"] > 200].copy()

# Filter cells with too many detected genes.
# These may be doublets or unusually large/high-RNA cells.
adata = adata[adata.obs["n_genes_by_counts"] < 6000].copy()

# Filter cells with high mitochondrial percentage.
# These may be stressed or damaged cells.
adata = adata[adata.obs["pct_counts_mt"] < 10].copy()

# Filter genes detected in fewer than 3 cells.
# These genes are too rare to help much in clustering.
sc.pp.filter_genes(adata, min_cells=3)

# Print filtering result
print("Filtering results:")
print(f"Cells before filtering: {n_cells_before}")
print(f"Cells after filtering:  {adata.n_obs}")
print(f"Cells removed:          {n_cells_before - adata.n_obs}")
print()
print(f"Genes before filtering: {n_genes_before}")
print(f"Genes after filtering:  {adata.n_vars}")
print(f"Genes removed:          {n_genes_before - adata.n_vars}")

# Save QC-filtered AnnData
qc_output_path = r"data/processed/gse150903_qc_filtered.h5ad"

adata.write_h5ad(qc_output_path)

print("Saved QC-filtered file to:", qc_output_path)

In [ ]:
##NORMALISATION OF DATA POINTS.

###Before normalization: A cell with more captured RNA can look artificially more active.

###After normalization: Cells are put on a comparable scale.

###After log transform: Expression values become easier to model and visualize.


# Save raw counts in a layer before normalization.
# This preserves the original count values inside the object.
adata.layers["counts"] = adata.X.copy()

# Normalize each cell so every cell has the same total count.
# This makes cells more comparable to each other.
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform the normalized counts.
# This reduces the impact of very large expression values.
sc.pp.log1p(adata)

# Store the normalized/log-transformed data as raw for plotting and marker analysis later.
adata.raw = adata

# Find highly variable genes.
# These are genes that vary across cells and help distinguish cell populations.
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat"
)

print("Highly variable genes:", adata.var["highly_variable"].sum())

# Save normalized object
normalized_output_path = r"data/processed/gse150903_normalized.h5ad"

adata.write_h5ad(normalized_output_path)

print("Saved normalized file to:", normalized_output_path)

In [ ]:
#INTERPRETATION OR DATA VISUALISATION

##PCA finds the major patterns in gene expression.
##Neighbors finds which cells are similar to each other.
##UMAP draws the cells as a 2D map.
##Clustering groups similar cells together.

import scanpy as sc

# Load the normalized AnnData object
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Keep only the highly variable genes.
# These are the 2,000 genes most useful for finding cell populations.
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

# Scale each gene so genes are comparable to each other.
# This prevents very high-expression genes from dominating PCA.
sc.pp.scale(adata_hvg, max_value=10)

# Run PCA.
# PCA compresses 2,000 genes into a smaller set of summary dimensions.
sc.tl.pca(adata_hvg, svd_solver="arpack")

# Plot how much information each PCA dimension explains.
# This helps us decide how many PCs to use.
sc.pl.pca_variance_ratio(adata_hvg, log=True)

# Build the cell-neighbor graph.
# This connects each cell to similar cells based on PCA.
sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=30)

# Run UMAP.
# UMAP creates a 2D map where similar cells appear near each other.
sc.tl.umap(adata_hvg)

# Cluster cells using Leiden clustering.
# Resolution controls how many clusters are created.
sc.tl.leiden(
    adata_hvg,
    resolution=0.5,
    flavor="igraph",
    n_iterations=2,
    directed=False
)

# Plot the UMAP colored by sample and by cluster.
sc.pl.umap(adata_hvg, color=["sample", "leiden"])

# Save the clustered object
clustered_output_path = r"data/processed/gse150903_clustered.h5ad"

adata_hvg.write_h5ad(clustered_output_path)

print("Saved clustered file to:", clustered_output_path)

In [ ]:
# Cell: Inspect UMAP Clusters And Sample Composition

# Count how many cells are in each Leiden cluster
print("Number of cells in each Leiden cluster:")
display(
    adata_hvg.obs["leiden"]
    .value_counts()
    .sort_index()
)

# Count how many cells from each sample are in each Leiden cluster
print("Number of cells from each sample in each Leiden cluster:")
cluster_sample_counts = pd.crosstab(
    adata_hvg.obs["leiden"],
    adata_hvg.obs["sample"]
)

display(cluster_sample_counts)

# Convert the table to percentages.
# This shows, for each cluster, what percent of cells came from each sample.
print("Percentage of each cluster coming from each sample:")
cluster_sample_percent = pd.crosstab(
    adata_hvg.obs["leiden"],
    adata_hvg.obs["sample"],
    normalize="index"
) * 100

display(cluster_sample_percent.round(2))

# Plot UMAP again, colored by biological sample and Leiden cluster
sc.pl.umap(
    adata_hvg,
    color=["sample", "leiden"],
    wspace=0.4
)

In [ ]:
# Cell: Find Marker Genes For Leiden Clusters : Wilcoxon method

import scanpy as sc
import pandas as pd

# Load clustered data
adata_hvg = sc.read_h5ad(
    r"data/processed/gse150903_clustered.h5ad"
)

# Run marker gene test.
# For each Leiden cluster, compare cells in that cluster against all other cells.
sc.tl.rank_genes_groups(
    adata_hvg,
    groupby="leiden",
    method="wilcoxon"
)

# Show a ranked marker-gene table inside the notebook.
# n_genes=10 means show the top 10 marker genes per cluster.
sc.pl.rank_genes_groups(
    adata_hvg,
    n_genes=10,
    sharey=False
)

# Convert marker-gene results into a regular pandas table.
marker_table = sc.get.rank_genes_groups_df(
    adata_hvg,
    group=None
)

# Show the first rows of the marker-gene table.
display(marker_table.head(20))

# Save marker genes to CSV so you can inspect them later.
marker_output_path = r"data/processed/gse150903_leiden_marker_genes.csv"

marker_table.to_csv(marker_output_path, index=False)

print("Saved marker gene table to:", marker_output_path)

In [ ]:
# Cell: Memory-Efficient Marker Gene Test

import scanpy as sc
import pandas as pd
from scipy import sparse

# Load normalized data, before scaling/PCA.
# This is better for marker genes than using scaled data.
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Load clustered data only to get the Leiden labels.
adata_hvg = sc.read_h5ad(
    r"data/processed/gse150903_clustered.h5ad"
)

# Copy Leiden cluster labels onto the normalized object.
adata.obs["leiden"] = adata_hvg.obs["leiden"]

# Convert expression matrix to sparse format if it is not already sparse.
# Sparse format saves memory because most single-cell expression values are zero.
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)

# Optional but useful: test only highly variable genes first.
# This reduces the number of genes tested from ~23,900 to 2,000.
adata_marker = adata[:, adata.var["highly_variable"]].copy()

# Run a faster marker-gene test first.
# t-test_overestim_var is less heavy than Wilcoxon and useful for exploration.
sc.tl.rank_genes_groups(
    adata_marker,
    groupby="leiden",
    method="t-test_overestim_var",
    n_genes=50
)

# Plot top marker genes per cluster.
sc.pl.rank_genes_groups(
    adata_marker,
    n_genes=10,
    sharey=False
)

# Convert results to a table.
marker_table = sc.get.rank_genes_groups_df(
    adata_marker,
    group=None
)

# Save marker table.
marker_output_path = r"data/processed/gse150903_marker_genes_memory_efficient.csv"

marker_table.to_csv(marker_output_path, index=False)

print("Saved marker gene table to:", marker_output_path)
display(marker_table.head(20))

In [ ]:
# Cell: find marker genes
#Question: For each cluster, which genes have higher average expression inside the cluster than outside it?
##This is a marker-gene screen, not a formal statistical test.

##for each leiden cluster, the code:
###1. Looks at all selected genes
###2. Calculates expression inside the cluster
###3. Calculates expression outside the cluster
###4. Calculates how much higher each gene is inside the cluster
###5. Sorts genes from strongest to weakest
###6. Keeps the top 50

# Cell: Memory-Efficient Marker Gene Screening Without Scanpy rank_genes_groups

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Load normalized data
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Load clustered data to get Leiden labels
adata_hvg = sc.read_h5ad(
    r"data/processed/gse150903_clustered.h5ad"
)

# Copy Leiden cluster labels onto normalized object
adata.obs["leiden"] = adata_hvg.obs["leiden"].astype(str)

# Use only highly variable genes for this first marker screen
adata_marker = adata[:, adata.var["highly_variable"]].copy()

# Make sure the matrix is sparse for memory efficiency
if not sparse.issparse(adata_marker.X):
    adata_marker.X = sparse.csr_matrix(adata_marker.X)

X = adata_marker.X.tocsr()
genes = adata_marker.var_names.to_numpy()
clusters = sorted(adata_marker.obs["leiden"].unique(), key=lambda x: int(x))

marker_rows = []

for cluster in clusters:
    in_cluster = (adata_marker.obs["leiden"].to_numpy() == cluster)
    out_cluster = ~in_cluster

    X_in = X[in_cluster]
    X_out = X[out_cluster]

    mean_in = np.asarray(X_in.mean(axis=0)).ravel()
    mean_out = np.asarray(X_out.mean(axis=0)).ravel()

    pct_in = X_in.getnnz(axis=0) / X_in.shape[0] * 100
    pct_out = X_out.getnnz(axis=0) / X_out.shape[0] * 100

    log2fc = np.log2((np.expm1(mean_in) + 1e-9) / (np.expm1(mean_out) + 1e-9))

    cluster_table = pd.DataFrame({
        "cluster": cluster,
        "gene": genes,
        "mean_in_cluster": mean_in,
        "mean_out_cluster": mean_out,
        "pct_in_cluster": pct_in,
        "pct_out_cluster": pct_out,
        "log2_fold_change": log2fc,
    })

    cluster_table = cluster_table.sort_values(
        ["log2_fold_change", "pct_in_cluster"],
        ascending=False
    ).head(50)

    marker_rows.append(cluster_table)

marker_table = pd.concat(marker_rows, ignore_index=True)

marker_output_path = r"data/processed/gse150903_marker_gene_screen_sparse.csv"

marker_table.to_csv(marker_output_path, index=False)

display(marker_table.head(30))

print("Saved marker screen to:", marker_output_path)

In [ ]:
##VISUALISE KNOWN MARKERS

known_markers = [
    "TTR", "AQP1", "KRT8", "KRT18", "OTX2",
    "SOX2", "PAX6", "VIM", "NES",
    "FOXJ1", "MKI67", "TOP2A", "DCX", "MAP2"
]

sc.pl.umap(
    adata_hvg,
    color=[gene for gene in known_markers if gene in adata_hvg.var_names],
    cmap="viridis"
)

In [ ]:
# Cell: Plot Known Marker Genes Across Leiden Clusters

import scanpy as sc
import pandas as pd

# Load the full normalized object.
# This contains all genes, which is better for checking known marker genes.
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Load the clustered object.
# This contains Leiden clusters and UMAP coordinates.
adata_hvg = sc.read_h5ad(
    r"data/processed/gse150903_clustered.h5ad"
)

# Copy Leiden cluster labels from the clustered object to the full normalized object.
adata.obs["leiden"] = adata_hvg.obs["leiden"].copy()

# Copy UMAP coordinates from the clustered object to the full normalized object.
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"].copy()

# Known marker genes for broad cell types/states expected in this dataset.
marker_sets = {
    "Choroid plexus epithelial": ["TTR", "AQP1", "KRT8", "KRT18", "OTX2", "CLDN2", "FOLR1"],
    "Neural progenitor": ["SOX2", "PAX6", "VIM", "NES", "HES1", "FABP7"],
    "Cycling cells": ["MKI67", "TOP2A", "CENPF", "PCNA", "STMN1"],
    "Neuron-like": ["DCX", "MAP2", "TUBB3", "STMN2", "RBFOX3"],
    "Ciliated/ependymal": ["FOXJ1", "PIFO", "TPPP3", "DNAH5", "RSPH1"],
    "Mesenchymal": ["COL1A1", "COL1A2", "DCN", "LUM", "VIM"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "CLDN5"],
}

# Keep only marker genes that are actually present in this dataset.
markers_present = {
    cell_type: [gene for gene in genes if gene in adata.var_names]
    for cell_type, genes in marker_sets.items()
}

# Print which markers are present and which are missing.
for cell_type, genes in marker_sets.items():
    present = [gene for gene in genes if gene in adata.var_names]
    missing = [gene for gene in genes if gene not in adata.var_names]
    print(f"\n{cell_type}")
    print("Present:", present)
    print("Missing:", missing)

# Dotplot:
# Dot size shows percent of cells in the cluster expressing the gene.
# Dot color shows average expression level.
sc.pl.dotplot(
    adata,
    var_names=markers_present,
    groupby="leiden",
    standard_scale="var",
    dendrogram=False
)

# UMAP view for a smaller set of key genes.
# This shows where marker genes are expressed across the cell map.
key_genes = [
    "TTR", "AQP1", "KRT8", "OTX2",
    "SOX2", "PAX6", "VIM",
    "MKI67", "TOP2A",
    "DCX", "MAP2",
    "FOXJ1"
]

key_genes_present = [gene for gene in key_genes if gene in adata.var_names]

sc.pl.umap(
    adata,
    color=["sample", "leiden"] + key_genes_present,
    cmap="viridis",
    ncols=3
)

In [ ]:
# Cell: Dotplot Marker Genes By Sample

import scanpy as sc

# Load normalized data
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Marker genes you want to compare across samples
marker_genes = [
    "TTR", "AQP1", "KRT8", "KRT18", "OTX2", "CLDN2", "FOLR1",
    "SOX2", "PAX6", "VIM", "NES", "HES1", "FABP7",
    "MKI67", "TOP2A", "CENPF", "PCNA",
    "DCX", "MAP2", "TUBB3",
    "FOXJ1", "PIFO", "TPPP3"
]

# Keep only genes that are actually present in your dataset
marker_genes_present = [gene for gene in marker_genes if gene in adata.var_names]

print("Genes included in dotplot:")
print(marker_genes_present)

print("\nGenes not found in dataset:")
print([gene for gene in marker_genes if gene not in adata.var_names])

# Create dotplot grouped by sample
sc.pl.dotplot(
    adata,
    var_names=marker_genes_present,
    groupby="sample",
    standard_scale="var",
    dendrogram=False
)

In [ ]:
import scanpy as sc

# Load normalized data
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)
marker_sets_from_paper = {
    "Choroid plexus": ["CLIC6", "HTR2C", "TTR", "AQP1"],
    "Telencephalon / neuronal": ["DCX", "FOXG1", "GAP43"],
    "Immature ChP / hem": ["MSX1", "OTX2", "RSPO3", "PAX6"],
    "Mature ChP epithelium": ["TTR", "KRT18", "NME5"],
    "ChP stroma / mesenchymal": ["COL1A1", "LUM", "DCN", "DLK1"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN", "INADL", "MPDZ"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Lateral ventricle ChP identity": ["LY6E"],
    "Third ventricle ChP identity": ["INS"],
    "Fourth ventricle ChP identity": ["PENK"],
    "Light / ciliated ChP epithelial": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP epithelial": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Dividing / cycling cells": ["MKI67", "TOP2A", "PCNA"],
}


# Print which markers are present and which are missing.
for cell_type, genes in marker_sets.items():
    present = [gene for gene in genes if gene in adata.var_names]
    missing = [gene for gene in genes if gene not in adata.var_names]
    print(f"\n{cell_type}")
    print("Present:", present)
    print("Missing:", missing)

sc.pl.dotplot(
    adata,
    var_names=marker_genes_present,
    groupby="sample",
    standard_scale="var",
    dendrogram=False
)

In [ ]:
# Cell: Dotplot Grouped Marker Genes By Sample

import scanpy as sc
import pandas as pd

# Load the full normalized object.
# This contains all genes and sample metadata.
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Marker genes grouped by biological category.
# These groups come from the paper and related ChP biology.
marker_sets_from_paper = {
    "Choroid plexus": ["CLIC6", "HTR2C", "TTR", "AQP1"],
    "Immature ChP / hem": ["MSX1", "OTX2", "RSPO3", "PAX6"],
    "Mature ChP epithelium": ["TTR", "KRT18", "NME5"],
    "ChP stroma / mesenchymal": ["COL1A1", "LUM", "DCN", "DLK1"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Telencephalon / neuronal": ["DCX", "FOXG1", "GAP43"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Dividing / cycling": ["MKI67", "TOP2A", "PCNA"],
}

# Keep only genes that are actually present in the dataset.
marker_sets_present = {
    group: [gene for gene in genes if gene in adata.var_names]
    for group, genes in marker_sets_from_paper.items()
}

# Remove marker groups that have no genes present.
marker_sets_present = {
    group: genes
    for group, genes in marker_sets_present.items()
    if len(genes) > 0
}

# Print which genes will be plotted in each group.
print("Marker groups included in plot:")
for group, genes in marker_sets_present.items():
    print(f"{group}: {genes}")

# Print missing genes, so you know what was not found.
print("\nMissing marker genes:")
for group, genes in marker_sets_from_paper.items():
    missing = [gene for gene in genes if gene not in adata.var_names]
    if missing:
        print(f"{group}: {missing}")

# Dotplot grouped by sample.
# Rows = samples
# Columns = marker genes
# Top brackets/labels = marker gene categories
# Dot size = percent of cells in that sample expressing the gene
# Dot color = average expression level
sc.pl.dotplot(
    adata,
    var_names=marker_sets_present,
    groupby="sample",
    standard_scale="var",
    dendrogram=False
)

In [ ]:
# Cell: Dotplot Grouped Marker Genes By Leiden Cluster

import scanpy as sc
import pandas as pd

# Load full normalized object
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Load clustered object to get Leiden labels and UMAP coordinates
adata_hvg = sc.read_h5ad(
    r"data/processed/gse150903_clustered.h5ad"
)

# Copy Leiden labels and UMAP coordinates onto the full normalized object
adata.obs["leiden"] = adata_hvg.obs["leiden"].copy()
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"].copy()

# Marker genes grouped by biological category
marker_sets_from_paper = {
    "Choroid plexus": ["CLIC6", "HTR2C", "TTR", "AQP1"],
    "Immature ChP / hem": ["MSX1", "OTX2", "RSPO3", "PAX6"],
    "Mature ChP epithelium": ["TTR", "KRT18", "NME5"],
    "ChP stroma / mesenchymal": ["COL1A1", "LUM", "DCN", "DLK1"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Telencephalon / neuronal": ["DCX", "FOXG1", "GAP43"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Dividing / cycling": ["MKI67", "TOP2A", "PCNA"],
}

# Keep only genes present in the dataset
marker_sets_present = {
    group: [gene for gene in genes if gene in adata.var_names]
    for group, genes in marker_sets_from_paper.items()
}

# Remove empty groups
marker_sets_present = {
    group: genes
    for group, genes in marker_sets_present.items()
    if len(genes) > 0
}

# Dotplot grouped by Leiden cluster
sc.pl.dotplot(
    adata,
    var_names=marker_sets_present,
    groupby="leiden",
    standard_scale="var",
    dendrogram=False
)

In [ ]:
# Cell: Automatically Annotate Clusters Using Marker Gene Sets

import scanpy as sc
import pandas as pd
import re

# Load full normalized object
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Load clustered object to get Leiden labels and UMAP
adata_hvg = sc.read_h5ad(
    r"data/processed/gse150903_clustered.h5ad"
)

adata.obs["leiden"] = adata_hvg.obs["leiden"].copy()
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"].copy()

# Your marker groups
marker_sets_from_paper = {
    "Choroid plexus": ["CLIC6", "HTR2C", "TTR", "AQP1"],
    "Immature ChP / hem": ["MSX1", "OTX2", "RSPO3", "PAX6"],
    "Mature ChP epithelium": ["TTR", "KRT18", "NME5"],
    "ChP stroma / mesenchymal": ["COL1A1", "LUM", "DCN", "DLK1"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Telencephalon / neuronal": ["DCX", "FOXG1", "GAP43"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Dividing / cycling": ["MKI67", "TOP2A", "PCNA"],
}

# Keep only genes present in your dataset
marker_sets_present = {
    cell_type: [gene for gene in genes if gene in adata.var_names]
    for cell_type, genes in marker_sets_from_paper.items()
}

# Remove empty marker groups
marker_sets_present = {
    cell_type: genes
    for cell_type, genes in marker_sets_present.items()
    if len(genes) > 0
}

# Score each cell for each marker group
score_columns = {}

for cell_type, genes in marker_sets_present.items():
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", cell_type).strip("_")
    score_col = f"score_{safe_name}"
    score_columns[cell_type] = score_col

    sc.tl.score_genes(
        adata,
        gene_list=genes,
        score_name=score_col
    )

# Average marker scores per Leiden cluster
cluster_scores = adata.obs.groupby

In [ ]:
# Cell: Automatically Annotate Clusters Using Marker Gene Sets With Mixed Review Label

import scanpy as sc
import pandas as pd
import re

# Load full normalized object
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Load clustered object to get Leiden labels and UMAP
adata_hvg = sc.read_h5ad(
    r"data/processed/gse150903_clustered.h5ad"
)

adata.obs["leiden"] = adata_hvg.obs["leiden"].copy()
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"].copy()

# Marker groups
marker_sets_from_paper = {
    "Choroid plexus": ["CLIC6", "HTR2C", "TTR", "AQP1"],
    "Immature ChP / hem": ["MSX1", "OTX2", "RSPO3", "PAX6"],
    "Mature ChP epithelium": ["TTR", "KRT18", "NME5"],
    "ChP stroma / mesenchymal": ["COL1A1", "LUM", "DCN", "DLK1"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Telencephalon / neuronal": ["DCX", "FOXG1", "GAP43"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Dividing / cycling": ["MKI67", "TOP2A", "PCNA"],
}

# Keep only marker genes present in the dataset
marker_sets_present = {
    cell_type: [gene for gene in genes if gene in adata.var_names]
    for cell_type, genes in marker_sets_from_paper.items()
}

# Remove empty marker groups
marker_sets_present = {
    cell_type: genes
    for cell_type, genes in marker_sets_present.items()
    if len(genes) > 0
}

# Score each cell for each marker group
score_columns = {}

for cell_type, genes in marker_sets_present.items():
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", cell_type).strip("_")
    score_col = f"score_{safe_name}"
    score_columns[cell_type] = score_col

    sc.tl.score_genes(
        adata,
        gene_list=genes,
        score_name=score_col
    )

# Average marker scores per Leiden cluster
cluster_scores = adata.obs.groupby("leiden")[list(score_columns.values())].mean()

# Rename score columns to readable cell-type names
cluster_scores = cluster_scores.rename(
    columns={v: k for k, v in score_columns.items()}
)

# Create annotation table
cluster_annotation = pd.DataFrame({
    "leiden": cluster_scores.index,
    "best_cell_type": cluster_scores.idxmax(axis=1),
    "best_score": cluster_scores.max(axis=1),
    "second_best_cell_type": cluster_scores.apply(
        lambda row: row.sort_values(ascending=False).index[1],
        axis=1
    ),
    "second_best_score": cluster_scores.apply(
        lambda row: row.sort_values(ascending=False).iloc[1],
        axis=1
    ),
})

# Calculate how much better the top label is than the second-best label
cluster_annotation["score_margin"] = (
    cluster_annotation["best_score"] - cluster_annotation["second_best_score"]
)

# If the score margin is small, label the cluster as mixed-underreview.
# You can adjust this threshold.
margin_threshold = 0.10

cluster_annotation["automatic_cell_type"] = cluster_annotation.apply(
    lambda row: "mixed-underreview"
    if row["score_margin"] < margin_threshold
    else row["best_cell_type"],
    axis=1
)

# Add automatic labels back to adata
cluster_to_cell_type = dict(
    zip(cluster_annotation["leiden"], cluster_annotation["automatic_cell_type"])
)

adata.obs["auto_cell_type"] = adata.obs["leiden"].map(cluster_to_cell_type)

# Save annotation table and scores to Excel
excel_output_path = r"data/processed/gse150903_auto_cluster_annotation.xlsx"

with pd.ExcelWriter(excel_output_path) as writer:
    cluster_annotation.to_excel(writer, sheet_name="auto_annotation", index=False)
    cluster_scores.to_excel(writer, sheet_name="cluster_marker_scores")

# Plot UMAP with automatic labels
sc.pl.umap(
    adata,
    color=["sample", "leiden", "auto_cell_type"],
    wspace=0.4
)

# Save annotated object
annotated_output_path = r"data/processed/gse150903_auto_annotated.h5ad"

adata.write_h5ad(annotated_output_path)

print("Margin threshold:", margin_threshold)
print("Clusters labeled mixed-underreview:")
display(
    cluster_annotation[
        cluster_annotation["automatic_cell_type"] == "mixed-underreview"
    ]
)

print("\nSaved automatic annotation Excel file to:")
print(excel_output_path)

print("\nSaved automatically annotated AnnData object to:")
print(annotated_output_path)

display(cluster_annotation)

In [ ]:
# Cell: Import Reviewed Cluster Annotations And Add To AnnData

import scanpy as sc
import pandas as pd

# Load full normalized object
adata = sc.read_h5ad(
    r"data/processed/gse150903_normalized.h5ad"
)

# Load clustered object to get Leiden labels and UMAP coordinates
adata_hvg = sc.read_h5ad(
    r"data/processed/gse150903_clustered.h5ad"
)

adata.obs["leiden"] = adata_hvg.obs["leiden"].copy()
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"].copy()

# Path to your reviewed Excel annotation file
annotation_excel_path = r"data/processed/gse150903_auto_cluster_annotation.xlsx"

# Read the reviewed annotation sheet
annotation_table = pd.read_excel(
    annotation_excel_path,
    sheet_name="auto_annotation"
)

# Show the first few rows so you can confirm the file loaded correctly
display(annotation_table.head())

# Use the reviewed label column.
# If you edited a different column in Excel, change this name.
final_label_column = "Reviewed_cell_type"

# Make sure Leiden cluster IDs are strings in both places
annotation_table["leiden"] = annotation_table["leiden"].astype(str)
adata.obs["leiden"] = adata.obs["leiden"].astype(str)

# Create a dictionary: cluster number -> reviewed cell type
cluster_to_cell_type = dict(
    zip(
        annotation_table["leiden"],
        annotation_table[final_label_column]
    )
)

# Add reviewed labels to every cell
adata.obs["cell_type_reviewed"] = adata.obs["leiden"].map(cluster_to_cell_type)

# Check if any cells did not get a label
missing_labels = adata.obs["cell_type_reviewed"].isna().sum()
print("Missing reviewed labels:", missing_labels)

# Plot UMAP with reviewed labels
sc.pl.umap(
    adata,
    color=["sample", "leiden", "cell_type_reviewed"],
    wspace=0.4
)

# Save reviewed annotated AnnData object
reviewed_output_path = r"data/processed/gse150903_reviewed_annotated.h5ad"

adata.write_h5ad(reviewed_output_path)

print("Saved reviewed annotated AnnData object to:")
print(reviewed_output_path)

In [ ]:
# Cell: Cell Type Composition By Sample

import pandas as pd
import matplotlib.pyplot as plt

# Count cells of each reviewed cell type in each sample
cell_type_counts = pd.crosstab(
    adata.obs["sample"],
    adata.obs["cell_type_reviewed"]
)

print("Cell counts by sample and reviewed cell type:")
display(cell_type_counts)

# Convert counts to percentages within each sample
cell_type_percent = pd.crosstab(
    adata.obs["sample"],
    adata.obs["cell_type_reviewed"],
    normalize="index"
) * 100

print("Cell type percentages by sample:")
display(cell_type_percent.round(2))

# Plot stacked bar chart of cell-type percentages
ax = cell_type_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    width=0.8
)

ax.set_ylabel("Percent of cells")
ax.set_xlabel("Sample")
ax.set_title("Reviewed Cell-Type Composition By Sample")
ax.legend(
    title="Reviewed cell type",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Cell: Marker Dotplot By Reviewed Cell Type

marker_sets_from_paper = {
    "Choroid plexus": ["CLIC6", "HTR2C", "TTR", "AQP1"],
    "Immature ChP / hem": ["MSX1", "OTX2", "RSPO3", "PAX6"],
    "Mature ChP epithelium": ["TTR", "KRT18", "NME5"],
    "ChP stroma / mesenchymal": ["COL1A1", "LUM", "DCN", "DLK1"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Telencephalon / neuronal": ["DCX", "FOXG1", "GAP43"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Dividing / cycling": ["MKI67", "TOP2A", "PCNA"],
}

marker_sets_present = {
    group: [gene for gene in genes if gene in adata.var_names]
    for group, genes in marker_sets_from_paper.items()
}

marker_sets_present = {
    group: genes
    for group, genes in marker_sets_present.items()
    if len(genes) > 0
}

sc.pl.dotplot(
    adata,
    var_names=marker_sets_present,
    groupby="cell_type_reviewed",
    standard_scale="var",
    dendrogram=False
)